# Facial Verification with a Siamese Network

This notebook implements **Facial Verification using a Siamese Network**.

The model learns to distinguish between two faces by comparing their embeddings using L1 distance. It outputs a score: 1 = same person, 0 = different person.

**Architecture Overview:**
- Embedding model: 4 Conv2D blocks → Dense(4096)
- L1 Distance layer: |embed1 - embed2|
- Siamese model: Two shared-weight embeddings → L1 distance → sigmoid classifier

# 1. Setup

## 1.1 Install Dependencies

> **UPDATE (TF 2.4.1 → 2.13.0):** TF 2.4.1 is from 2021 and uses deprecated APIs (`tf.config.experimental`). TF 2.13 is the last stable unified CPU+GPU release — the `tensorflow-gpu` package is no longer a separate install from TF 2.x onwards.

In [ ]:
# [UPDATED] Removed tensorflow-gpu — no longer needed as a separate package from TF 2.x
# [UPDATED] Pinned to TF 2.13.0 — last stable unified CPU+GPU release
# opencv-python-headless preferred in notebook environments (no GUI display dependencies)
!pip install tensorflow==2.13.0 opencv-python-headless matplotlib

## 1.2 Import Dependencies

In [ ]:
# Standard library
import cv2
import os
import uuid
import random
import numpy as np
from matplotlib import pyplot as plt

# TensorFlow and Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
from tensorflow.keras.metrics import Precision, Recall

print(f"TensorFlow version: {tf.__version__}")

## 1.3 Set Random Seeds & GPU Growth

> **UPDATE:** Added `tf.random.set_seed()` and `np.random.seed()` for reproducibility.
> **UPDATE:** `tf.config.experimental.list_physical_devices` is deprecated — replaced with the stable `tf.config.list_physical_devices` (available since TF 2.2).

In [ ]:
# Set random seeds for reproducibility across runs
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# [UPDATED] Use tf.config.list_physical_devices (stable) instead of tf.config.experimental.*
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    # set_memory_growth is still under tf.config.experimental (not yet promoted)
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    print(f"GPUs available: {[g.name for g in gpus]}")
else:
    print("No GPU detected — running on CPU.")

## 1.4 Create Folder Structures

In [ ]:
# Dataset paths
POS_PATH = os.path.join('data', 'positive')   # Same-person images (webcam)
NEG_PATH = os.path.join('data', 'negative')   # Different-person images (LFW dataset)
ANC_PATH = os.path.join('data', 'anchor')     # Anchor/reference images (webcam)

# [UPDATED] Added exist_ok=True — prevents FileExistsError when re-running the notebook
os.makedirs(POS_PATH, exist_ok=True)
os.makedirs(NEG_PATH, exist_ok=True)
os.makedirs(ANC_PATH, exist_ok=True)

print("Folder structure ready:")
for p in [ANC_PATH, POS_PATH, NEG_PATH]:
    print(f"  {p}/")

# 2. Collect Positives and Anchors

Two data sources:
- **Anchor + Positive**: Your own face via webcam → label = 1 (same person)
- **Negative**: LFW dataset (other people's faces) → label = 0 (different person)

## 2.1 Untar Labelled Faces in the Wild Dataset

LFW dataset: http://vis-www.cs.umass.edu/lfw/

Download `lfw.tgz` from the above URL and place it in the project root before running.

In [ ]:
# Extract LFW archive — run only once
# Download from: http://vis-www.cs.umass.edu/lfw/lfw.tgz
!tar -xf lfw.tgz

In [ ]:
# Move all LFW images into data/negative/ (flatten nested person-name folders)
# LFW structure: lfw/<PersonName>/<image>.jpg
# Note: os.replace() silently overwrites if two people share a filename (rare with 13k+ images)
moved = 0
for directory in os.listdir('lfw'):
    person_dir = os.path.join('lfw', directory)
    if not os.path.isdir(person_dir):
        continue
    for file in os.listdir(person_dir):
        src = os.path.join(person_dir, file)
        dst = os.path.join(NEG_PATH, file)
        os.replace(src, dst)
        moved += 1

print(f"Moved {moved} images to {NEG_PATH}")

## 2.2 Collect Anchor and Positive Images via Webcam

**Controls while the window is open:**
- Press `a` → save current frame as an **Anchor** image
- Press `p` → save current frame as a **Positive** image
- Press `q` → quit

> **UPDATE:** Changed `VideoCapture(4)` → `VideoCapture(0)`. Index 4 was hard-coded to the original machine. Most systems use index 0 for the built-in camera.

In [ ]:
# [UPDATED] Use index 0 (standard built-in webcam). Change if your camera is on a different index.
WEBCAM_INDEX = 0

cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(f"Could not open webcam at index {WEBCAM_INDEX}. Try index 1 or 2.")

anchor_count = 0
positive_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Failed to read frame — exiting.")
        break

    # Crop a 250x250 region — matches what the model was trained on
    frame = frame[120:120+250, 200:200+250, :]

    # 'a' = capture anchor image
    if cv2.waitKey(1) & 0xFF == ord('a'):
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)
        anchor_count += 1
        print(f"Anchor saved ({anchor_count} total)")

    # 'p' = capture positive image
    if cv2.waitKey(1) & 0xFF == ord('p'):
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)
        positive_count += 1
        print(f"Positive saved ({positive_count} total)")

    cv2.imshow('Collect — A=anchor  P=positive  Q=quit', frame)

    # 'q' = quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Done. Anchors: {anchor_count} | Positives: {positive_count}")

## 2.3 Data Augmentation

Augment each captured image to generate 9 variants, boosting dataset size.

**Augmentations applied:**
1. Random brightness
2. Random contrast
3. Random crop (re-enabled from original)
4. Random horizontal flip
5. Random JPEG quality
6. Random saturation

> **FIX (Seed Bug):** Original code used fixed `seed=(1,2)` for brightness and `seed=(1,3)` for contrast in ALL 9 iterations — meaning every augmented image got the exact same brightness and contrast change. Seeds are now randomised per iteration so each variant is truly different.

> **RE-ENABLED:** `stateless_random_crop` was commented out. It's now active — spatial augmentation improves robustness to framing differences.

In [ ]:
def data_aug(img):
    """
    Apply random augmentations to a single image, returning 9 variants.

    Args:
        img: NumPy array or tensor of shape (H, W, 3), uint8.

    Returns:
        List of 9 augmented uint8 tensors.
    """
    data = []
    img = tf.cast(img, tf.uint8)

    for _ in range(9):
        # [FIX] Generate unique random seeds per iteration per augmentation
        # Original: all used seed=(1,2) or seed=(1,3) — identical across all 9 iterations
        def rnd_seed():
            return (np.random.randint(0, 1000), np.random.randint(0, 1000))

        aug = img

        # [FIX] Randomised seed — was fixed seed=(1,2) for all iterations
        aug = tf.image.stateless_random_brightness(aug, max_delta=0.02, seed=rnd_seed())

        # [FIX] Randomised seed — was fixed seed=(1,3) for all iterations
        aug = tf.image.stateless_random_contrast(aug, lower=0.6, upper=1.0, seed=rnd_seed())

        # [RE-ENABLED] Random crop then resize back — was commented out in original
        # Adds spatial robustness (handles slight framing variation)
        aug = tf.image.stateless_random_crop(aug, size=(200, 200, 3), seed=rnd_seed())
        aug = tf.image.resize(aug, (250, 250))
        aug = tf.cast(aug, tf.uint8)

        aug = tf.image.stateless_random_flip_left_right(aug, seed=rnd_seed())
        aug = tf.image.stateless_random_jpeg_quality(
            aug, min_jpeg_quality=90, max_jpeg_quality=100, seed=rnd_seed()
        )
        aug = tf.image.stateless_random_saturation(aug, lower=0.9, upper=1.0, seed=rnd_seed())

        data.append(aug)

    return data

In [ ]:
# Augment all Anchor images
print("Augmenting anchor images...")
for file_name in os.listdir(ANC_PATH):
    img = cv2.imread(os.path.join(ANC_PATH, file_name))
    if img is None:
        continue
    for aug_img in data_aug(img):
        cv2.imwrite(os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1())), aug_img.numpy())
print(f"Anchor total: {len(os.listdir(ANC_PATH))} images")

# Augment all Positive images
print("Augmenting positive images...")
for file_name in os.listdir(POS_PATH):
    img = cv2.imread(os.path.join(POS_PATH, file_name))
    if img is None:
        continue
    for aug_img in data_aug(img):
        cv2.imwrite(os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1())), aug_img.numpy())
print(f"Positive total: {len(os.listdir(POS_PATH))} images")

# 3. Load and Preprocess Images

Build a `tf.data` pipeline: list file paths → pair as anchor/positive/negative → preprocess on the fly → split 70/30 train/test.

## 3.1 Get Image Directories

In [ ]:
# List image paths and cap at 3000 per class for balanced training
# [NOTE] .take(3000) silently takes all available if fewer than 3000 exist
# The count printout below confirms actual loaded sizes
anchor   = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg').take(3000)
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg').take(3000)
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg').take(3000)

# Sanity check: print actual counts
print(f"Anchor images:   {len(list(anchor))}")
print(f"Positive images: {len(list(positive))}")
print(f"Negative images: {len(list(negative))}")

# Re-create datasets after consuming them for the count
anchor   = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg').take(3000)
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg').take(3000)
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg').take(3000)

## 3.2 Preprocessing — Scale and Resize

In [ ]:
def preprocess(file_path):
    """
    Load an image from disk and prepare it for the model.

    Steps:
      1. Read raw bytes from file path
      2. Decode JPEG (force 3 channels — no alpha)
      3. Resize to 100x100 (model input size)
      4. Normalise to [0, 1] float32

    Args:
        file_path: String tensor — path to a .jpg image file.

    Returns:
        float32 tensor of shape (100, 100, 3) with values in [0, 1].
    """
    byte_img = tf.io.read_file(file_path)
    # channels=3 ensures RGB output regardless of original encoding
    img = tf.io.decode_jpeg(byte_img, channels=3)
    img = tf.image.resize(img, (100, 100))
    img = img / 255.0  # Normalise to [0, 1]
    return img

## 3.3 Create Labelled Dataset

In [ ]:
# Pair each anchor with a positive → label 1.0 (same person)
positives = tf.data.Dataset.zip((
    anchor,
    positive,
    tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))
))

# Pair each anchor with a negative → label 0.0 (different person)
negatives = tf.data.Dataset.zip((
    anchor,
    negative,
    tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))
))

# Combine both into one dataset
data = positives.concatenate(negatives)
print(f"Total pairs: {len(list(data))}")

## 3.4 Build Train and Test Partitions

In [ ]:
def preprocess_twin(input_img, validation_img, label):
    """
    Apply preprocess() to both images in a pair.
    Used as a .map() function over the dataset.

    Args:
        input_img:      File path tensor for the anchor image.
        validation_img: File path tensor for the positive/negative image.
        label:          Float tensor — 1.0 (same person) or 0.0 (different).

    Returns:
        Tuple of (preprocessed_input, preprocessed_validation, label).
    """
    return preprocess(input_img), preprocess(validation_img), label

In [ ]:
# Build the full data pipeline
data = data.map(preprocess_twin)                    # Preprocess image pairs on the fly
data = data.cache()                                 # Cache after first epoch (speeds up subsequent epochs)
data = data.shuffle(buffer_size=10_000, seed=SEED)  # Shuffle with fixed seed for reproducibility

# Compute split sizes
total      = len(list(data))
train_size = round(total * 0.7)
test_size  = total - train_size
print(f"Total: {total} | Train: {train_size} | Test: {test_size}")

# Training partition — 70%
train_data = data.take(train_size)
train_data = train_data.batch(16)
# [UPDATED] AUTOTUNE dynamically selects optimal prefetch buffer size at runtime
train_data = train_data.prefetch(tf.data.AUTOTUNE)

# Testing partition — remaining 30%
test_data = data.skip(train_size)
test_data = test_data.batch(16)
test_data = test_data.prefetch(tf.data.AUTOTUNE)

# 4. Model Engineering

Three components:
1. **Embedding model** — shared CNN mapping each face to a 4096-dim vector
2. **L1 Distance layer** — element-wise absolute difference between embeddings
3. **Siamese model** — wires both together with a sigmoid classifier

## 4.1 Build Embedding Model

Inspired by Koch et al. (2015) — *Siamese Neural Networks for One-shot Image Recognition*.

```
Input(100,100,3)
  → Conv2D(64,  10×10, relu) → MaxPool(2×2)
  → Conv2D(128,  7×7, relu) → MaxPool(2×2)
  → Conv2D(128,  4×4, relu) → MaxPool(2×2)
  → Conv2D(256,  4×4, relu)
  → Flatten → Dense(4096, sigmoid)
```

In [ ]:
def make_embedding():
    """
    Build the shared CNN embedding model.

    Both branches of the Siamese network use the SAME instance (shared weights),
    so both images are projected into the same feature space.

    Returns:
        Keras Model — Input: (100,100,3)  Output: (4096,)
    """
    inp = Input(shape=(100, 100, 3), name='input_image')

    # Block 1 — large 10x10 kernel captures coarse facial structure
    c1 = Conv2D(64, (10, 10), activation='relu')(inp)
    m1 = MaxPooling2D(64, (2, 2), padding='same')(c1)

    # Block 2 — 7x7 kernel extracts mid-level features
    c2 = Conv2D(128, (7, 7), activation='relu')(m1)
    m2 = MaxPooling2D(64, (2, 2), padding='same')(c2)

    # Block 3 — 4x4 kernel focuses on fine-grained local features
    c3 = Conv2D(128, (4, 4), activation='relu')(m2)
    m3 = MaxPooling2D(64, (2, 2), padding='same')(c3)

    # Block 4 — high-capacity final conv before projection
    c4 = Conv2D(256, (4, 4), activation='relu')(m3)

    # Project flattened features into 4096-dim embedding space
    # Sigmoid activation keeps all embedding values in [0, 1]
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation='sigmoid')(f1)

    return Model(inputs=[inp], outputs=[d1], name='embedding')

embedding = make_embedding()
embedding.summary()

## 4.2 Build L1 Distance Layer

L1 (Manhattan) distance: `d(a, b) = |a - b|` applied element-wise over all 4096 dimensions.
The subsequent `Dense(1, sigmoid)` learns which dimensions matter most for identity.

In [ ]:
class L1Dist(Layer):
    """
    Custom Keras layer computing element-wise L1 (Manhattan) distance.

    Formula: |input_embedding - validation_embedding|

    Output shape: same as input — (batch, 4096)
    The Dense(1, sigmoid) classifier after this layer learns a weighted
    combination of these distances to output a final match score.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, input_embedding, validation_embedding):
        """
        Args:
            input_embedding:      Tensor (batch, 4096) — embedding of anchor image.
            validation_embedding: Tensor (batch, 4096) — embedding of validation image.

        Returns:
            Tensor (batch, 4096) — element-wise absolute difference.
        """
        return tf.math.abs(input_embedding - validation_embedding)

## 4.3 Build Siamese Model

```
input_image ──────► embedding ──┐
                                ├──► L1Dist ──► Dense(1, sigmoid) ──► match score
validation_image ──► embedding ──┘
```
Both branches share the same `embedding` model instance (shared weights).

In [ ]:
def make_siamese_model():
    """
    Build the full Siamese verification network.

    Takes two images and outputs a single score:
      - Score > 0.5  →  same person
      - Score <= 0.5 →  different person

    Returns:
        Keras Model with two (100,100,3) inputs and one scalar sigmoid output.
    """
    input_image      = Input(name='input_img',      shape=(100, 100, 3))
    validation_image = Input(name='validation_img', shape=(100, 100, 3))

    # Both images pass through the SAME embedding model (shared weights)
    inp_embedding = embedding(input_image)
    val_embedding = embedding(validation_image)

    # Compute L1 distance between the two embedding vectors
    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer(inp_embedding, val_embedding)

    # Binary classifier: 1 = same person, 0 = different
    classifier = Dense(1, activation='sigmoid')(distances)

    return Model(
        inputs=[input_image, validation_image],
        outputs=classifier,
        name='SiameseNetwork'
    )

siamese_model = make_siamese_model()
siamese_model.summary()

# 5. Training

We use a **custom training loop** with `tf.GradientTape` instead of `model.fit()` for explicit control over each gradient step.

## 5.1 Setup Loss and Optimizer

In [ ]:
# Binary Cross-Entropy measures separation between same-pair (1) and different-pair (0)
binary_cross_loss = tf.losses.BinaryCrossentropy()

# Adam with lr=1e-4 — standard choice for fine-grained vision tasks
# Lower to 1e-5 if training loss is unstable
opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

## 5.2 Establish Checkpoints

In [ ]:
checkpoint_dir    = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(optimizer=opt, siamese_model=siamese_model)

os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {checkpoint_dir}/")

## 5.3 Train Step Function

In [ ]:
@tf.function
def train_step(batch):
    """
    Perform one gradient update on a single batch.

    Args:
        batch: Tuple of (anchor_imgs, validation_imgs, labels)
               Image shapes: (batch_size, 100, 100, 3)
               Label shape:  (batch_size,)

    Returns:
        Scalar loss for this batch.
    """
    with tf.GradientTape() as tape:
        X = batch[:2]   # anchor and positive/negative image tensors
        y = batch[2]    # ground truth labels

        # Forward pass (training=True enables dropout/batchnorm if added later)
        yhat = siamese_model(X, training=True)
        loss = binary_cross_loss(y, yhat)

    # Compute gradients and update all trainable weights
    grad = tape.gradient(loss, siamese_model.trainable_variables)
    opt.apply_gradients(zip(grad, siamese_model.trainable_variables))

    return loss

## 5.4 Training Loop

In [ ]:
def train(data, EPOCHS):
    """
    Full training loop with per-epoch metrics and checkpoint saving.

    Args:
        data:   Batched tf.data.Dataset (train_data)
        EPOCHS: Number of training epochs

    Prints loss, recall, and precision at the end of each epoch.
    Saves a checkpoint every 10 epochs.
    """
    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        progbar = tf.keras.utils.Progbar(len(data))

        # Reset per-epoch metrics
        r = Recall()
        p = Precision()

        for idx, batch in enumerate(data):
            loss = train_step(batch)
            yhat = siamese_model.predict(batch[:2], verbose=0)
            r.update_state(batch[2], yhat)
            p.update_state(batch[2], yhat)
            progbar.update(idx + 1)

        recall_val    = r.result().numpy()
        precision_val = p.result().numpy()
        print(f"  Loss: {loss.numpy():.4f} | Recall: {recall_val:.4f} | Precision: {precision_val:.4f}")

        # Save checkpoint every 10 epochs
        if epoch % 10 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f"  Checkpoint saved at epoch {epoch}.")

## 5.5 Train the Model

In [ ]:
EPOCHS = 50
train(train_data, EPOCHS)

# 6. Evaluate Model

Evaluate on the held-out test set using **Recall** and **Precision**.

- **Recall** = TP / (TP + FN) — did we catch all same-person pairs?
- **Precision** = TP / (TP + FP) — when we predict 'same', are we right?

In [ ]:
# Evaluate across the full test set
r = Recall()
p = Precision()

for test_input, test_val, y_true in test_data.as_numpy_iterator():
    yhat = siamese_model.predict([test_input, test_val], verbose=0)
    r.update_state(y_true, yhat)
    p.update_state(y_true, yhat)

print(f"Test Recall:    {r.result().numpy():.4f}")
print(f"Test Precision: {p.result().numpy():.4f}")

## 6.1 Visualise a Prediction Pair

In [ ]:
# Grab one batch from the test set for visual inspection
test_input, test_val, y_true = test_data.as_numpy_iterator().next()
y_hat  = siamese_model.predict([test_input, test_val], verbose=0)
y_pred = [1 if score > 0.5 else 0 for score in y_hat]

print("Predicted:", y_pred[:8])
print("True:     ", list(y_true[:8].astype(int)))

# Show the first image pair in the batch
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(test_input[0])
axes[0].set_title('Input (Anchor)')
axes[0].axis('off')

axes[1].imshow(test_val[0])
label_text = 'Same Person' if y_pred[0] == 1 else 'Different Person'
correct    = y_pred[0] == int(y_true[0])
color      = 'green' if correct else 'red'
axes[1].set_title(f"Validation | Predicted: {label_text} ({'Correct' if correct else 'Wrong'})", color=color)
axes[1].axis('off')

plt.tight_layout()
plt.show()

# 7. Save and Load Model

> **UPDATE:** `.h5` format is deprecated in TF 2.x. Replaced with TF's native **SavedModel** format (folder), which correctly serialises custom layers and is the recommended format from TF 2.x onwards.

In [ ]:
# [UPDATED] Save as SavedModel directory (not .h5 — deprecated in TF 2.x)
# SavedModel handles custom layers like L1Dist more reliably than HDF5
SAVE_PATH = 'siamese_model_saved'
siamese_model.save(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}/")

In [ ]:
# Load the saved model — L1Dist must be passed as a custom object
siamese_model = tf.keras.models.load_model(
    SAVE_PATH,
    custom_objects={'L1Dist': L1Dist}
)

print("Model loaded successfully.")
siamese_model.summary()

# 8. Real-Time Verification

The verification system compares a live webcam frame against a folder of stored reference images.

**Two-threshold design:**
- `detection_threshold`: Minimum model score for a single image pair to count as a match (default 0.5)
- `verification_threshold`: Minimum proportion of reference images that must match for access (default 0.5)

This makes the system more robust — a single lucky match doesn't grant access.

In [ ]:
# Application data paths
APP_INPUT_PATH = os.path.join('application_data', 'input_image')
APP_VERIF_PATH = os.path.join('application_data', 'verification_images')

os.makedirs(APP_INPUT_PATH, exist_ok=True)
os.makedirs(APP_VERIF_PATH, exist_ok=True)

print(f"Input image folder:         {APP_INPUT_PATH}/")
print(f"Verification images folder: {APP_VERIF_PATH}/")
print(f"  → Place your reference face images in: {APP_VERIF_PATH}/")

In [ ]:
def verify(model, detection_threshold=0.5, verification_threshold=0.5):
    """
    Verify if the live input image matches the stored reference images.

    Steps:
      1. Load the current input image (saved by the webcam loop)
      2. Compare it against every image in the verification folder
      3. Count comparisons that exceed detection_threshold
      4. Verify if that proportion exceeds verification_threshold

    Args:
        model:                  Trained Siamese model.
        detection_threshold:    Score above which a pair counts as a match. Default: 0.5
        verification_threshold: Fraction of reference images that must match. Default: 0.5

    Returns:
        results:  List[float] — raw model scores for each reference image.
        verified: bool — True if the person is verified.
    """
    results = []
    input_img = preprocess(os.path.join(APP_INPUT_PATH, 'input_image.jpg'))

    verif_files = [
        f for f in os.listdir(APP_VERIF_PATH)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if not verif_files:
        print("Warning: No reference images found. Add images to the verification_images folder.")
        return [], False

    for image_name in verif_files:
        val_img = preprocess(os.path.join(APP_VERIF_PATH, image_name))
        result = model.predict(
            [np.expand_dims(input_img, axis=0), np.expand_dims(val_img, axis=0)],
            verbose=0
        )
        results.append(result[0][0])

    # Count matches above detection threshold
    detection = np.sum(np.array(results) > detection_threshold)

    # Compute verification ratio
    verification_ratio = detection / len(verif_files)
    verified = verification_ratio > verification_threshold

    print(f"Matches: {detection}/{len(verif_files)} | Ratio: {verification_ratio:.2f} | Verified: {verified}")
    return results, verified

## 8.1 Real-Time Webcam Verification

**Controls:**
- Press `v` → capture current frame and run verification
- Press `q` → quit

> **UPDATE:** Changed `VideoCapture(4)` → `VideoCapture(0)` (same fix as Section 2.2).
> **REMOVED:** Commented-out HSV brightness block — dead code, removed for cleanliness.

In [ ]:
# [UPDATED] VideoCapture(0) — was hardcoded to 4 (device-specific to original machine)
cap = cv2.VideoCapture(WEBCAM_INDEX)

if not cap.isOpened():
    raise RuntimeError(f"Could not open webcam at index {WEBCAM_INDEX}.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Frame read failed.")
        break

    # Crop to 250x250 — same region used during data collection
    frame = frame[120:120+250, 200:200+250, :]
    cv2.imshow('Verification — V=verify  Q=quit', frame)

    # 'v' = verify
    if cv2.waitKey(10) & 0xFF == ord('v'):
        # Save current frame as input image
        cv2.imwrite(os.path.join(APP_INPUT_PATH, 'input_image.jpg'), frame)
        # Run verification against stored references
        results, verified = verify(siamese_model, detection_threshold=0.5, verification_threshold=0.5)
        print(f"VERIFIED: {verified}")

    # 'q' = quit
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()